# EDA — Dataset SOME/IP IDS (Kim et al. 2026)

Análise exploratória do tráfego de rede a partir do `parsed_packets.csv`.

**Objetivo:** entender o protocolo SOME/IP, caracterizar cada tipo de ataque e o modelo de ameaça.

| Seção | Conteúdo |
|-------|----------|
| 1 | Visão geral e distribuição |
| 2 | Protocolo SOME/IP — estrutura e fluxos |
| 3 | Tráfego normal — o que é "bom" |
| 4 | DoS — Notification Flood |
| 5 | Fuzzy — payload aleatório |
| 6 | MITM — intercepção e replay |
| 7 | Modelo de ameaça — tabela consolidada |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#161b2e',
    'axes.edgecolor':   '#2a3550',
    'axes.labelcolor':  '#c9d1e0',
    'xtick.color':      '#6a7a9a',
    'ytick.color':      '#6a7a9a',
    'text.color':       '#c9d1e0',
    'grid.color':       '#1e2840',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#161b2e',
    'legend.edgecolor': '#2a3550',
    'font.size':        11,
})

COLORS = {
    'normal': '#7bff9c',
    'dos':    '#ff7b7b',
    'fuzzy':  '#ffd97b',
    'mitm':   '#c07bff',
}

# ── Ajuste o caminho conforme seu ambiente ────────────────────────────────
CSV = r'C:\Mestrado\SDV_Research\experiments\notebooks\output_parser\parsed_packets.csv'
# CSV = '/content/processed/parsed_packets.csv'   # Colab

CHUNK = 500_000
print('Configuração OK')

## 1. Visão Geral e Distribuição

In [ ]:
# Leitura leve — só colunas de interesse
LIGHT_COLS = ['label', 'transport', 'is_sd', 'msg_type',
              'service_id', 'ip_len', 'someip_payload_len',
              'src_ip', 'dst_ip', 'src_port', 'dst_port', 'timestamp']

df = pd.read_csv(CSV, usecols=LIGHT_COLS, low_memory=False)
print(f'Total de registros: {len(df):,}')
print(f'Colunas carregadas: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── 1.1 Distribuição por classe ───────────────────────────────────────────
counts = df['label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Distribuição de Pacotes por Classe', color='white', fontsize=13)

# Barras
ax = axes[0]
bars = ax.bar(counts.index, counts.values,
              color=[COLORS[l] for l in counts.index], alpha=0.85)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30_000,
            f'{val/1e6:.2f}M', ha='center', va='bottom', fontsize=10, color='#c9d1e0')
ax.set_ylabel('Pacotes'); ax.set_title('Contagem absoluta')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.grid(axis='y')

# Pizza
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=counts.index,
    colors=[COLORS[l] for l in counts.index],
    autopct='%1.1f%%', startangle=140,
    textprops={'color': '#c9d1e0'},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2}
)
for at in autotexts: at.set_color('#0f1117'); at.set_fontweight('bold')
ax2.set_title('Proporção (%)')

plt.tight_layout()
plt.savefig('eda_01_distribuicao.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print(counts.to_string())

In [ ]:
# ── 1.2 TCP vs UDP e is_sd por classe ────────────────────────────────────
summary = df.groupby('label').agg(
    total        = ('label', 'count'),
    pct_tcp      = ('transport',  lambda x: f"{(x=='TCP').mean()*100:.1f}%"),
    pct_udp      = ('transport',  lambda x: f"{(x=='UDP').mean()*100:.1f}%"),
    pct_sd       = ('is_sd',      lambda x: f"{x.fillna(False).astype(bool).mean()*100:.1f}%"),
    ip_len_mean  = ('ip_len',            'mean'),
    payload_mean = ('someip_payload_len','mean'),
    payload_std  = ('someip_payload_len','std'),
    payload_max  = ('someip_payload_len','max'),
)
summary['ip_len_mean']  = summary['ip_len_mean'].round(1)
summary['payload_mean'] = summary['payload_mean'].round(1)
summary['payload_std']  = summary['payload_std'].round(1)
print(summary.to_string())

## 2. Protocolo SOME/IP — Serviços e Fluxos

In [ ]:
# ── 2.1 Service IDs ───────────────────────────────────────────────────────
SERVICE_NAMES = {
    4097.0: 'GPS (0x1001)',
    4098.0: 'IMU (0x1002)',
    4099.0: 'VDE (0x1003)',
    65535.0: 'SD (0xFFFF)',
}
df['service_name'] = df['service_id'].map(SERVICE_NAMES).fillna('Outro')

pivot = df.groupby(['label','service_name']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')
pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='#0f1117', linewidth=0.5)
ax.set_title('Pacotes por Classe × Serviço SOME/IP', color='white')
ax.set_xlabel('Classe')
ax.set_ylabel('Pacotes')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))
ax.set_xticklabels(pivot.index, rotation=0)
ax.legend(loc='upper right')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('eda_02_servicos.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('\nProporção por serviço dentro de cada classe:')
print(pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1).to_string())

In [ ]:
# ── 2.2 IPs únicos por classe (topologia de rede) ─────────────────────────
print('=== IPs de ORIGEM únicos por classe ===')
for label in ['normal', 'dos', 'fuzzy', 'mitm']:
    sub = df[df['label'] == label]
    print(f'\n[{label.upper()}] — {len(sub):,} pacotes')
    print(sub['src_ip'].value_counts().head(8).to_string())

print('\n=== Pares (src_ip, dst_ip) únicos por classe ===')
for label in ['normal', 'dos', 'fuzzy', 'mitm']:
    sub = df[df['label'] == label]
    n_pairs = sub.groupby(['src_ip', 'dst_ip']).ngroups
    print(f'{label:<8}: {n_pairs:>4} pares únicos')

## 3. Tráfego Normal — o que caracteriza tráfego benigno

In [ ]:
normal = df[df['label'] == 'normal'].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Tráfego NORMAL — Distribuições', color='white', fontsize=13)

# Payload length
normal['someip_payload_len'].dropna().astype(int).hist(
    ax=axes[0], bins=30, color='#7bff9c', alpha=0.8, edgecolor='#0f1117')
axes[0].set_title('Payload length (bytes)')
axes[0].set_xlabel('bytes'); axes[0].grid(True)

# IP length
normal['ip_len'].hist(
    ax=axes[1], bins=30, color='#7bff9c', alpha=0.8, edgecolor='#0f1117')
axes[1].set_title('IP frame length (bytes)')
axes[1].set_xlabel('bytes'); axes[1].grid(True)

# Serviços
svc_counts = normal['service_name'].value_counts()
axes[2].barh(svc_counts.index, svc_counts.values, color='#7bff9c', alpha=0.8)
axes[2].set_title('Serviços (normal)')
axes[2].set_xlabel('pacotes'); axes[2].grid(axis='x')

plt.tight_layout()
plt.savefig('eda_03_normal.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('Payload (normal):')
print(normal['someip_payload_len'].describe().round(2))

In [ ]:
# ── 3.1 Fluxos normais — pares IP:porta ──────────────────────────────────
print('Top 15 fluxos no tráfego NORMAL (src_ip → dst_ip:dst_port):')
flow_normal = normal.groupby(['src_ip','dst_ip','dst_port','service_name']).size()\
                    .sort_values(ascending=False).head(15)
print(flow_normal.to_string())

## 4. DoS — Notification Flood

O atacante envia um volume massivo de notificações SOME/IP para um serviço específico,
esgotando recursos do receptor (buffer overflow, CPU).

In [ ]:
dos = df[df['label'] == 'dos'].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Tráfego DoS — Distribuições', color='white', fontsize=13)

# Payload
dos['someip_payload_len'].dropna().astype(int).hist(
    ax=axes[0], bins=30, color='#ff7b7b', alpha=0.8, edgecolor='#0f1117')
axes[0].set_title('Payload length (bytes)')
axes[0].set_xlabel('bytes'); axes[0].grid(True)

# IP len
dos['ip_len'].hist(ax=axes[1], bins=30, color='#ff7b7b', alpha=0.8, edgecolor='#0f1117')
axes[1].set_title('IP frame length')
axes[1].set_xlabel('bytes'); axes[1].grid(True)

# Serviços
svc_counts = dos['service_name'].value_counts()
axes[2].barh(svc_counts.index, svc_counts.values, color='#ff7b7b', alpha=0.8)
axes[2].set_title('Serviços (DoS)')
axes[2].set_xlabel('pacotes'); axes[2].grid(axis='x')

plt.tight_layout()
plt.savefig('eda_04_dos.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('Proporção SD (DoS):', f"{(dos['is_sd'].fillna(False).astype(bool).mean()*100):.1f}%")
print('Payload stats (DoS):', dos['someip_payload_len'].describe().round(2))

In [ ]:
# ── 4.1 DoS: fluxos mais ativos (atacante → vítima) ──────────────────────
print('Top 10 fluxos no tráfego DoS (src_ip → dst_ip:service):')
flow_dos = dos.groupby(['src_ip','dst_ip','service_name']).size()\
               .sort_values(ascending=False).head(10)
print(flow_dos.to_string())

print('\nComparação de src_ip: aparecem IPs novos no DoS?')
normal_ips = set(df[df['label']=='normal']['src_ip'].unique())
dos_ips    = set(dos['src_ip'].unique())
print(f'IPs somente em normal: {normal_ips - dos_ips}')
print(f'IPs somente em DoS:    {dos_ips - normal_ips}')
print(f'IPs em ambos:          {normal_ips & dos_ips}')

## 5. Fuzzy — Payload Aleatório

O atacante injeta payloads com conteúdo aleatório (fuzzing) para tentar causar
comportamentos inesperados ou crash no ECU receptor.

In [ ]:
fuzzy = df[df['label'] == 'fuzzy'].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Tráfego Fuzzy — Distribuições', color='white', fontsize=13)

# Payload — note a cauda longa
fuzzy['someip_payload_len'].dropna().clip(upper=200).astype(int).hist(
    ax=axes[0], bins=50, color='#ffd97b', alpha=0.8, edgecolor='#0f1117')
axes[0].set_title('Payload length (bytes, clip@200)')
axes[0].set_xlabel('bytes'); axes[0].grid(True)

# IP len
fuzzy['ip_len'].clip(upper=400).hist(
    ax=axes[1], bins=50, color='#ffd97b', alpha=0.8, edgecolor='#0f1117')
axes[1].set_title('IP frame length')
axes[1].set_xlabel('bytes'); axes[1].grid(True)

# Serviços
svc_counts = fuzzy['service_name'].value_counts()
axes[2].barh(svc_counts.index, svc_counts.values, color='#ffd97b', alpha=0.8)
axes[2].set_title('Serviços (Fuzzy)')
axes[2].set_xlabel('pacotes'); axes[2].grid(axis='x')

plt.tight_layout()
plt.savefig('eda_05_fuzzy.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('Payload stats (Fuzzy):')
print(fuzzy['someip_payload_len'].describe().round(2))
print(f'\nPayloads > 100 bytes: {(fuzzy["someip_payload_len"]>100).sum():,} '
      f'({(fuzzy["someip_payload_len"]>100).mean()*100:.1f}%)')

In [ ]:
# ── 5.1 Comparação de entropia de payload: normal vs fuzzy ───────────────
def byte_entropy(hex_str):
    """Shannon entropy dos bytes do payload."""
    if not isinstance(hex_str, str) or len(hex_str) < 2:
        return np.nan
    try:
        raw = bytes.fromhex(hex_str)
    except ValueError:
        return np.nan
    if len(raw) == 0:
        return 0.0
    counts = np.bincount(np.frombuffer(raw, dtype=np.uint8), minlength=256)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

# Amostra de 5000 pacotes por classe para não travar
SAMPLE = 5_000
rng = np.random.default_rng(42)

entropy_data = {}
for lbl in ['normal', 'dos', 'fuzzy', 'mitm']:
    sub = df[df['label'] == lbl]['someip_payload_hex'].dropna()
    if len(sub) > SAMPLE:
        sub = sub.iloc[rng.choice(len(sub), SAMPLE, replace=False)]
    entropy_data[lbl] = sub.apply(byte_entropy).dropna().values

fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor('#0f1117')
for lbl, vals in entropy_data.items():
    ax.hist(vals, bins=40, alpha=0.6, label=lbl, color=COLORS[lbl], density=True)
ax.set_title('Entropia de Shannon do payload SOME/IP por classe', color='white')
ax.set_xlabel('Entropia (bits)')
ax.set_ylabel('Densidade')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('eda_06_entropia.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('Entropia média por classe:')
for lbl, vals in entropy_data.items():
    print(f'  {lbl:<8}: {np.mean(vals):.3f} bits  (std={np.std(vals):.3f})')

## 6. MITM — Intercepção e Replay

O atacante intercepta mensagens legítimas e as reenvia (replay),
podendo também modificar o conteúdo. O payload tende a ser idêntico
ou muito similar ao tráfego normal (baixa distância de Hamming vs. pacote anterior).

In [ ]:
mitm = df[df['label'] == 'mitm'].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Tráfego MITM — Distribuições', color='white', fontsize=13)

# Payload
mitm['someip_payload_len'].dropna().astype(int).hist(
    ax=axes[0], bins=30, color='#c07bff', alpha=0.8, edgecolor='#0f1117')
axes[0].set_title('Payload length (bytes)')
axes[0].set_xlabel('bytes'); axes[0].grid(True)

# IP len
mitm['ip_len'].hist(ax=axes[1], bins=30, color='#c07bff', alpha=0.8, edgecolor='#0f1117')
axes[1].set_title('IP frame length')
axes[1].set_xlabel('bytes'); axes[1].grid(True)

# Serviços
svc_counts = mitm['service_name'].value_counts()
axes[2].barh(svc_counts.index, svc_counts.values, color='#c07bff', alpha=0.8)
axes[2].set_title('Serviços (MITM)')
axes[2].set_xlabel('pacotes'); axes[2].grid(axis='x')

plt.tight_layout()
plt.savefig('eda_07_mitm.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f'MITM — proporção SD: {(mitm["is_sd"].fillna(False).astype(bool).mean()*100):.1f}%')
print('MITM — IPs ativos:')
print(mitm['src_ip'].value_counts().head(8).to_string())

## 7. Comparação Direta — Normal vs Ataques

In [ ]:
# ── 7.1 Boxplot de payload por classe ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparação Normal vs Ataques', color='white', fontsize=13)

labels_ord = ['normal', 'dos', 'mitm', 'fuzzy']
data_payload = [df[df['label']==l]['someip_payload_len'].dropna().values for l in labels_ord]
data_iplen   = [df[df['label']==l]['ip_len'].values for l in labels_ord]

for ax, data, title in [
    (axes[0], data_payload, 'SOME/IP Payload length (bytes)'),
    (axes[1], data_iplen,   'IP frame length (bytes)'),
]:
    bp = ax.boxplot(data, labels=labels_ord, patch_artist=True,
                    medianprops={'color':'white','linewidth':2},
                    whiskerprops={'color':'#5a6a88'},
                    capprops={'color':'#5a6a88'},
                    flierprops={'marker':'o','markersize':2,'color':'#5a6a88','alpha':0.3})
    for patch, lbl in zip(bp['boxes'], labels_ord):
        patch.set_facecolor(COLORS[lbl])
        patch.set_alpha(0.7)
    ax.set_title(title)
    ax.set_ylim(0, None)
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig('eda_08_boxplot.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 7.2 Proporção SD e transporte por classe ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('TCP vs UDP  e  SOME/IP-SD por classe', color='white', fontsize=13)

for ax, col, title, vals in [
    (axes[0], 'transport', 'TCP vs UDP',
     df.groupby('label')['transport'].value_counts(normalize=True).unstack(fill_value=0)),
    (axes[1], 'is_sd', '% SOME/IP-SD (UDP)',
     df.groupby('label')['is_sd'].apply(lambda x: x.fillna(False).astype(bool).mean()).rename('SD')
       .to_frame()),
]:
    vals.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='#0f1117',
              linewidth=0.5, legend=(col=='transport'))
    ax.set_title(title)
    ax.set_xticklabels(vals.index, rotation=0)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig('eda_09_transport_sd.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 7.3 Tabela consolidada: modelo de ameaça ──────────────────────────────
threat_model = {
    'normal': {
        'Vetor':         'GPS/IMU/VDE broadcasts periódicos',
        'Transporte':    '~99% TCP, ~1% UDP/SD',
        'Serviços':      '0x1001/1002/1003',
        'Payload':       'Pequeno (23 B), muito consistente (std 7 B)',
        'Entropia':      'Baixa — estrutura previsível',
        'Taxa':          'Periódica / estável',
        'Assinatura':    'Fluxos fixos, payload repetitivo',
    },
    'dos': {
        'Vetor':         'Flood de NOTIFICATION para serviço-alvo',
        'Transporte':    '84% TCP + 16% UDP/SD (descoberta)',
        'Serviços':      'IMU (0x1002) mais atingido',
        'Payload':       'Ligeiramente maior (29 B), std 12 B',
        'Entropia':      'Média — payloads repetitivos mas em volume',
        'Taxa':          'Alta / rajadas',
        'Assinatura':    'Muitos pacotes → mesmo destino (mesma porta/serviço)',
    },
    'fuzzy': {
        'Vetor':         'Injeção de payloads aleatórios (fuzzing)',
        'Transporte':    '99% TCP',
        'Serviços':      'IMU (maior), GPS, VDE — todos alvos',
        'Payload':       'Alta variação (std 52 B, max 1332 B)',
        'Entropia':      'ALTA — bytes aleatórios',
        'Taxa':          'Alta / constante',
        'Assinatura':    'Payload size muito variável + entropia alta',
    },
    'mitm': {
        'Vetor':         'Intercepção + replay de mensagens legítimas',
        'Transporte':    '85% TCP + 15% UDP/SD (monitoramento)',
        'Serviços':      'Todos — monitoramento amplo',
        'Payload':       'Similar ao normal (29 B) — mensagens replicadas',
        'Entropia':      'Baixa-média — payloads legítimos replicados',
        'Taxa':          'Normal ou levemente elevada',
        'Assinatura':    'Alto % SD (monitoramento) + mesmo payload repetido (Hamming≈0)',
    },
}

import textwrap
print('=' * 100)
print('MODELO DE AMEAÇA — SOME/IP IDS')
print('=' * 100)
fields = ['Vetor','Transporte','Serviços','Payload','Entropia','Taxa','Assinatura']
header = f'{"":<15}' + ''.join(f'{l:<26}' for l in ['NORMAL','DOS','FUZZY','MITM'])
print(header)
print('-' * 100)
for f in fields:
    row = f'{f:<15}'
    for lbl in ['normal','dos','fuzzy','mitm']:
        val = threat_model[lbl].get(f,'')
        row += f'{val[:25]:<26}'
    print(row)
print('=' * 100)

In [ ]:
# ── 7.4 Feature discriminability visual ──────────────────────────────────
# Mostra quais dimensões separam melhor as classes
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Separabilidade: features chave por classe', color='white', fontsize=13)

for ax, (feat, title, clip_val) in zip(axes.flat, [
    ('someip_payload_len', 'Payload length (bytes)', 150),
    ('ip_len',             'IP frame length (bytes)', 300),
    ('someip_payload_len', 'Payload length — zoom', 60),
    ('ip_len',             'IP length — zoom', 100),
]):
    for lbl in ['normal','dos','fuzzy','mitm']:
        vals = df[df['label']==lbl][feat].dropna().clip(upper=clip_val)
        ax.hist(vals, bins=40, alpha=0.5, label=lbl,
                color=COLORS[lbl], density=True)
    ax.set_title(title); ax.set_xlabel(feat); ax.set_ylabel('Densidade')
    ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout()
plt.savefig('eda_10_separabilidade.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 8. Resumo — O que cada feature captura

| Feature | Normal | DoS | Fuzzy | MITM |
|---------|--------|-----|-------|------|
| f01 IP time interval | baixo, periódico | muito baixo (flood) | baixo-médio | normal |
| f02 SOME/IP likelihood | alto (payload conhecido) | alto | **muito baixo** | alto |
| f03 SOME/IP-SD likelihood | alto | médio | médio | baixo-médio |
| f04 TCP/UDP likelihood | alto | alto | **muito baixo** | alto |
| f05 SOME/IP entropy | baixa | baixa | **muito alta** | baixa |
| f06 SOME/IP-SD entropy | baixa | média | média | média |
| f07 TCP/UDP entropy | baixa | baixa | **muito alta** | baixa |
| f08 SOME/IP payload changes | baixo (períodico) | médio | **muito alto** | **baixo** (replay=0) |
| f09 SOME/IP-SD payload changes | baixo | alto | alto | alto |
| f10 TCP/UDP payload changes | baixo | médio | **muito alto** | **muito baixo** (replay) |
| f11 IP length changes | baixo | médio | **muito alto** | baixo |
| f12 TCP/UDP length changes | baixo | médio | **muito alto** | baixo |

**Assinaturas discriminantes:**
- **Fuzzy**: alta entropia + alto payload changes + payload size variável → features f05, f07, f08, f10, f11, f12 explodem
- **MITM**: payload changes ≈ 0 (replay exato) + alto SD → f08/f10 próximos de zero, mas f09 alto
- **DoS**: f01 (time interval) muito baixo (flood) + f09 alto (SD ativo)
- **Normal**: tudo baixo e periódico